# DFU Classification — Project Overview

> **Goal**: Classify Diabetic Foot Ulcer (DFU) images using Deep Learning  
> **Dataset**: INAOE (334 images: CT=90, DM=244)  
> **Framework**: TensorFlow 2.21 + Keras 3  
> **GPU**: NVIDIA Blackwell (RTX 5060 Ti, sm_120a)

| Label | Class | Description |
|-------|-------|-------------|
| CT    | 0     | Control (normal wound) |
| DM    | 1     | Diabetic wound |

## 1. Dataset & Splits

```
334 images (224x224 px, normalized [0,1], .npy format)
├── Test Set  (~67 images, 20%)  — held out, evaluated once at the end
└── Train+Val (~267 images, 80%)
    ├── Fold 1 (train ~213 / val ~54)
    ├── Fold 2
    ├── Fold 3
    ├── Fold 4
    └── Fold 5
```

- **Seed**: 42 (fixed across all scripts)
- **Stratified** split to preserve CT:DM ratio in every fold

In [ ]:
import sys, os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

warnings.filterwarnings('ignore')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

sys.path.insert(0, '.')
from dfu_common import CONFIG, load_preprocessed_inaoe, create_fold_splits

RESULTS_DIR = Path(CONFIG['results_dir'])
print('TF version:', end=' ')
import tensorflow as tf; print(tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU') or 'CPU only')

In [ ]:
X, y = load_preprocessed_inaoe()
print(f'Total images : {len(X)}')
print(f'CT  (label=0): {(y==0).sum()}')
print(f'DM  (label=1): {(y==1).sum()}')
print(f'Image shape  : {X.shape[1:]}')
print(f'Value range  : [{X.min():.3f}, {X.max():.3f}]')

In [ ]:
ct_idx = np.where(y == 0)[0][:4]
dm_idx = np.where(y == 1)[0][:4]

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
fig.suptitle('Sample Images — CT (top) vs DM (bottom)', fontsize=14)
for i, idx in enumerate(ct_idx):
    axes[0, i].imshow(X[idx])
    axes[0, i].set_title(f'CT [{idx}]', fontsize=10)
    axes[0, i].axis('off')
for i, idx in enumerate(dm_idx):
    axes[1, i].imshow(X[idx])
    axes[1, i].set_title(f'DM [{idx}]', fontsize=10)
    axes[1, i].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Train+Val: {len(X_trainval)}  (CT={(y_trainval==0).sum()}, DM={(y_trainval==1).sum()})')
print(f'Test set : {len(X_test)}  (CT={(y_test==0).sum()}, DM={(y_test==1).sum()})')

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
for split, (yl, title) in enumerate([(y_trainval, 'Train+Val'), (y_test, 'Test')]):
    counts = [(yl==0).sum(), (yl==1).sum()]
    ax[split].bar(['CT (0)', 'DM (1)'], counts, color=['steelblue', 'tomato'])
    ax[split].set_title(title)
    ax[split].set_ylabel('Count')
    for j, v in enumerate(counts):
        ax[split].text(j, v + 1, str(v), ha='center')
plt.suptitle('Class Distribution per Split')
plt.tight_layout()
plt.show()

## 2. Model Architecture

```
Input (224x224x3)
    |
Backbone (ImageNet pretrained, frozen in Phase 1)
    |
CBAM (reduction_ratio=16)
    |-- Channel Attention : AvgPool + MaxPool -> Shared MLP -> sigmoid
    +-- Spatial Attention : AvgPool + MaxPool along C -> Conv2D(7x7) -> sigmoid
    |
GlobalAveragePooling2D
    |
Dense(256, relu) -> Dropout(0.5)
    |
Dense(64, relu)  -> Dropout(0.5)
    |
Dense(1, sigmoid)
```

**Backbones tested**: EfficientNetB0, ResNet50, ConvNeXt-Tiny  
**Winner**: ResNet50 (highest AUC in RQ1)

### Two-Phase Training

| Phase | Backbone | Max Epochs | Early Stopping | LR |
|-------|----------|-----------|----------------|----|
| 1 | Frozen | 50 | patience=5 (val_loss) | 1e-3 |
| 2 | Unfreeze top 30% | 50 | patience=15 (val_loss) | 1e-4 (Exp decay) |

In [ ]:
from dfu_common import DFUModelTrainer

best_params = {
    'dropout_rate': 0.5,
    'l2_reg': 1e-4,
    'dense_units_1': 256,
    'dense_units_2': 64,
}

trainer = DFUModelTrainer('ResNet50', best_params)
trainer.build()
trainer.model.summary(line_length=80)

## 3. Hyperparameter Tuning — Optuna

| Parameter | Search Space |
|-----------|-------------|
| `dropout_rate` | 0.2 – 0.5 |
| `l2_reg` | 1e-5 – 1e-2 |
| `dense_units_1` | {128, 256, 512} |
| `dense_units_2` | {64, 128, 256} |
| `batch_size` | {16, 32} |
| `phase1_lr` | 1e-4 – 1e-2 |
| `phase2_lr` | 1e-6 – 1e-4 |

- **Sampler**: TPE (Tree-structured Parzen Estimator), Seed=42  
- **Trials**: 10 trials on Fold 1 only (for speed)

## 4. RQ1 — Backbone Comparison

**Objective**: Select the best backbone from 3 candidates using 5-fold CV on the training set.

**Clinical screening criteria** (all three must be met):
- AUC-ROC >= 0.80
- Sensitivity >= 0.85
- Specificity >= 0.70

In [ ]:
rq1_path = RESULTS_DIR / 'rq1_results.json'

if rq1_path.exists():
    with open(rq1_path) as f:
        rq1_raw = json.load(f)
    rows = []
    for backbone, metrics in rq1_raw.items():
        rows.append({
            'Backbone': backbone,
            'AUC': round(metrics.get('auc', 0), 4),
            'Sensitivity': round(metrics.get('sensitivity', 0), 4),
            'Specificity': round(metrics.get('specificity', 0), 4),
        })
    df_rq1 = pd.DataFrame(rows)
else:
    print('rq1_results.json not found — using values from PROJECT_OVERVIEW.md')
    df_rq1 = pd.DataFrame({
        'Backbone':    ['EfficientNetB0', 'ResNet50', 'ConvNeXt-Tiny'],
        'AUC':         [0.4500, 0.8235, 0.8207],
        'Sensitivity': [1.0000, 0.9846, 0.8359],
        'Specificity': [0.0000, 0.1981, 0.6257],
    })

df_rq1['AUC>=0.80']  = df_rq1['AUC'] >= 0.80
df_rq1['Sens>=0.85'] = df_rq1['Sensitivity'] >= 0.85
df_rq1['Spec>=0.70'] = df_rq1['Specificity'] >= 0.70
df_rq1['Pass All']   = df_rq1['AUC>=0.80'] & df_rq1['Sens>=0.85'] & df_rq1['Spec>=0.70']
display(df_rq1)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(df_rq1))
w = 0.25
colors = ['#4C72B0', '#DD8452', '#55A868']
for i, (m, c) in enumerate(zip(['AUC', 'Sensitivity', 'Specificity'], colors)):
    ax.bar(x + i*w, df_rq1[m], w, label=m, color=c)
for thr, c in zip([0.80, 0.85, 0.70], colors):
    ax.axhline(thr, color=c, linestyle='--', alpha=0.5, linewidth=1)
ax.set_xticks(x + w)
ax.set_xticklabels(df_rq1['Backbone'])
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('RQ1 — Backbone Comparison (5-fold CV, threshold=0.5)')
ax.legend()
plt.tight_layout()
plt.show()
print('Winner: ResNet50 (highest AUC = 0.8235)')

## 5. RQ2 — Threshold Optimization (Youden's Index)

**Objective**: Compare default threshold (0.5) against Youden's Index threshold on ResNet50+CBAM.

$$J = \text{Sensitivity} + \text{Specificity} - 1 \qquad \text{threshold}^* = \arg\max(\text{TPR} - \text{FPR})$$

In [ ]:
rq2_path = RESULTS_DIR / 'rq2_results.json'

if rq2_path.exists():
    with open(rq2_path) as f:
        rq2 = json.load(f)
    print(json.dumps(rq2, indent=2))
else:
    print('rq2_results.json not found — using values from PROJECT_OVERVIEW.md')

df_folds = pd.DataFrame({
    'Fold':       [1, 2, 3, 4, 5, 'Mean'],
    'Youden thr': [0.6754, 0.6990, 0.7006, 0.6401, 0.7847, 0.7000],
    'Sens':       [0.9231, 0.8974, 0.8462, 0.9231, 0.7436, None],
    'Spec':       [0.5333, 0.5333, 0.8571, 0.9286, 0.8571, None],
})
display(df_folds)

df_comp = pd.DataFrame({
    'Metric':          ['Sensitivity', 'Specificity'],
    'Default (0.5)':   [0.9846, 0.1981],
    'Youden (0.7000)': [0.7795, 0.7267],
    'Delta':           ['-0.2051 (-20.8%)', '+0.5286 (+266.9%)'],
})
print()
display(df_comp)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([0.5, 0.7], [0.9846, 0.7795], 'o-', label='Sensitivity', color='tomato')
ax.plot([0.5, 0.7], [0.1981, 0.7267], 's-', label='Specificity', color='steelblue')
ax.axhline(0.85, color='tomato',    linestyle='--', alpha=0.4, label='Sens criterion 0.85')
ax.axhline(0.70, color='steelblue', linestyle='--', alpha=0.4, label='Spec criterion 0.70')
ax.set_xticks([0.5, 0.7])
ax.set_xticklabels(['Default\n(0.5)', "Youden's\n(0.7)"])
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('RQ2 — Threshold Trade-off: Sensitivity vs Specificity')
ax.legend()
plt.tight_layout()
plt.show()

## 6. RQ3 — Final Evaluation on Test Set

**Strategy**:
1. Read average stopping epoch from 5-fold CV (stored in `ConvNeXt-Tiny_avg_epochs.json`)
2. Retrain on **full training set (267 images)** for that fixed number of epochs
3. No early stopping in the final retrain
4. Evaluate with Youden threshold (0.7146 from RQ2)

**Average stopping epochs** (ResNet50+CBAM): Phase 1 = **20**, Phase 2 = **49**

In [ ]:
rq3_path = RESULTS_DIR / 'rq3_results.json'

if rq3_path.exists():
    with open(rq3_path) as f:
        rq3 = json.load(f)
else:
    print('rq3_results.json not found — using values from PROJECT_OVERVIEW.md')
    rq3 = {
        'Sensitivity': 0.7551, 'Specificity': 0.6667,
        'AUC-ROC':     0.8447, 'PPV':         0.8605,
        'NPV':         0.5000, 'F1-Score':    0.8043,
        'Accuracy':    0.7313,
    }

display(pd.DataFrame({'Metric': list(rq3.keys()), 'Value': list(rq3.values())}))

In [ ]:
metrics = ['Sensitivity', 'Specificity', 'AUC-ROC', 'PPV', 'NPV', 'F1-Score', 'Accuracy']
values  = [rq3.get(m, 0) for m in metrics]

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['tomato' if v < 0.70 else 'steelblue' for v in values]
bars = ax.bar(metrics, values, color=colors)
ax.axhline(0.70, color='gray', linestyle='--', alpha=0.5, label='0.70 reference')
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.01, f'{v:.4f}', ha='center', fontsize=9)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('RQ3 — Final Test Set Evaluation (ResNet50+CBAM, Youden thr=0.7)')
ax.legend()
plt.tight_layout()
plt.show()

## 7. RQ4 — Explainability (Grad-CAM)

**Objective**: Visualize which image regions the model attends to.

| Method | Concept |
|--------|--------|
| **Grad-CAM** | Weight feature maps by global-avg-pooled gradients |
| **Grad-CAM++** | Weight by alpha coefficients from 2nd-order gradients |
| **Eigen-CAM** | Gradient-free — uses PC1 from SVD of the feature map |

**Output**: 4-panel image (Original / Grad-CAM / Grad-CAM++ / Eigen-CAM) saved to `results/rq4_gradcam/`

> **Note**: The **Top-Region Pointing Game** (which measures whether the highest-activation region overlaps with the true ROI) has not been conducted, as the INAOE dataset does not provide ground-truth ROI annotations. Grad-CAM evaluation here is therefore **qualitative only**.

In [ ]:
gradcam_dir = RESULTS_DIR / 'rq4_gradcam'

if gradcam_dir.exists():
    cam_images = sorted(gradcam_dir.glob('*.png'))[:6]
    if cam_images:
        from PIL import Image
        n = len(cam_images)
        fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
        if n == 1:
            axes = [axes]
        for ax, img_path in zip(axes, cam_images):
            ax.imshow(Image.open(img_path))
            ax.set_title(img_path.stem[:30], fontsize=8)
            ax.axis('off')
        plt.suptitle('RQ4 — Grad-CAM Visualizations')
        plt.tight_layout()
        plt.show()
    else:
        print('No PNG images found in results/rq4_gradcam/')
else:
    print('results/rq4_gradcam/ not found.')
    print('Run first: ./run_gpu.sh rq4_gradcam.py')

## 8. RQ5 — CNN vs BPNN

**Objective**: Compare the proposed CNN against a BPNN trained on handcrafted features.

### BPNN Feature Extraction (24-dim)

| Features | Details | Dim |
|----------|---------|-----|
| GLCM | 8-level quantized, 4 angles (0/45/90/135), 4 properties x 4 angles | 16 |
| HOG | 8x8 cells, 8 statistics (mean, std, var, median, max, min, skew, kurtosis) | 8 |
| **Total** | | **24** |

**Best BPNN**: architecture=(256, 128), activation=tanh, alpha=0.0001, Youden thr=0.5792  
**Statistical tests**: McNemar's Test + Bootstrap AUC p-value (2,000 samples)

In [ ]:
rq5_path = RESULTS_DIR / 'rq5_results.json'

if rq5_path.exists():
    with open(rq5_path) as f:
        rq5 = json.load(f)
    print(json.dumps(rq5, indent=2))
else:
    print('rq5_results.json not found — using values from PROJECT_OVERVIEW.md')

df_rq5 = pd.DataFrame({
    'Metric':            ['Sensitivity', 'Specificity', 'AUC-ROC', 'PPV', 'NPV', 'F1-Score'],
    'ResNet50+CBAM':     [0.7551, 0.6667, 0.8447, 0.8605, 0.5000, 0.8043],
    'BPNN (GLCM+HOG)':   [0.8367, 0.6111, 0.8526, 0.8542, 0.5789, 0.8454],
})
df_rq5['Delta'] = (df_rq5['BPNN (GLCM+HOG)'] - df_rq5['ResNet50+CBAM']).round(4)
display(df_rq5)

In [ ]:
metrics   = df_rq5['Metric'].tolist()
cnn_vals  = df_rq5['ResNet50+CBAM'].tolist()
bpnn_vals = df_rq5['BPNN (GLCM+HOG)'].tolist()

x, w = np.arange(len(metrics)), 0.35
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w/2, cnn_vals,  w, label='ResNet50+CBAM (CNN)', color='steelblue')
ax.bar(x + w/2, bpnn_vals, w, label='BPNN (GLCM+HOG)',     color='tomato')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('RQ5 — CNN vs BPNN Performance on Test Set')
ax.legend()
for i, (cv, bv) in enumerate(zip(cnn_vals, bpnn_vals)):
    ax.text(i - w/2, cv + 0.01, f'{cv:.3f}', ha='center', fontsize=8)
    ax.text(i + w/2, bv + 0.01, f'{bv:.3f}', ha='center', fontsize=8)
plt.tight_layout()
plt.show()

## 9. RQ6 — BPNN Feature Interpretability (Supplementary)

**Objective**: Understand which features drive the BPNN's decisions.

| Method | Concept | Output |
|--------|---------|--------|
| **Permutation Importance** | Shuffle each feature, measure AUC drop | bar chart |
| **SHAP KernelExplainer** | Shapley value per feature per sample | beeswarm + bar |

**Key findings**:
- **HOG Median and HOG Mean** are the most important — overall gradient magnitude is the primary signal
- DM feet show lower HOG values than CT — more uniform pressure distribution
- **Homogeneity 0° and Contrast 0°** are the most important GLCM features
- Correlation features have almost no effect on predictions

In [ ]:
rq6_dir = RESULTS_DIR / 'rq6_bpnn_interpretability'
plots = ['permutation_importance.png', 'shap_beeswarm.png', 'shap_bar.png']
found = [rq6_dir / p for p in plots if (rq6_dir / p).exists()]

if found:
    from PIL import Image
    fig, axes = plt.subplots(1, len(found), figsize=(7*len(found), 6))
    if len(found) == 1:
        axes = [axes]
    for ax, img_path in zip(axes, found):
        ax.imshow(Image.open(img_path))
        ax.set_title(img_path.stem.replace('_', ' ').title())
        ax.axis('off')
    plt.suptitle('RQ6 — BPNN Feature Interpretability')
    plt.tight_layout()
    plt.show()
else:
    print('RQ6 plots not found.')
    print('Run first: ./run_gpu.sh rq6_bpnn_interpretability.py')

## 10. Summary Dashboard

In [ ]:
titles     = ['RQ1: Best Backbone\n(ResNet50)',
               'RQ2: Youden thr=0.70',
               'RQ3: Test Set\n(ResNet50+CBAM)',
               'RQ5: CNN vs BPNN\n(BPNN shown)']
auc_vals   = [0.8235, 0.8235, 0.8447, 0.8526]
sens_vals  = [0.9846, 0.7795, 0.7551, 0.8367]
spec_vals  = [0.1981, 0.7267, 0.6667, 0.6111]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for i, ax in enumerate(axes):
    vals  = [auc_vals[i], sens_vals[i], spec_vals[i]]
    labs  = ['AUC', 'Sens', 'Spec']
    cols  = ['steelblue', 'tomato', 'seagreen']
    bars  = ax.bar(labs, vals, color=cols)
    ax.axhline(0.80, color='steelblue', linestyle=':', alpha=0.5)
    ax.axhline(0.85, color='tomato',    linestyle=':', alpha=0.5)
    ax.axhline(0.70, color='seagreen',  linestyle=':', alpha=0.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.01, f'{v:.3f}',
                ha='center', fontsize=9)
    ax.set_ylim(0, 1.1)
    ax.set_title(titles[i], fontsize=10)

plt.suptitle('Project Summary Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Running Order

```bash
# 1. Train backbones
./run_gpu.sh train_resnet.py
./run_gpu.sh train_efficientnet.py
./run_gpu.sh train_convnext.py

# 2. RQ1 — backbone selection
./run_gpu.sh rq1_backbone_comparison.py

# 3. RQ2 — threshold optimization
./run_gpu.sh rq2_threshold_optimization.py

# 4. RQ3 — final test evaluation
./run_gpu.sh rq3_final_evaluation.py

# 5. RQ4 — XAI visualizations
./run_gpu.sh rq4_gradcam.py

# 6. RQ5 — BPNN comparison
./run_gpu.sh rq5_bpnn_comparison.py

# 7. RQ6 — BPNN interpretability (run after RQ5)
./run_gpu.sh rq6_bpnn_interpretability.py
```

## Technical Notes

| Topic | Detail |
|-------|--------|
| Framework | TensorFlow 2.21 + Keras 3 |
| Checkpoint format | `.keras` (not `.h5`) |
| GPU | NVIDIA Blackwell (sm_120a) — requires `jit_compile=False` |
| Load model | `compile=False` + `custom_objects={'CBAM': CBAM}` |
| Augmentation | `RandomRotation(±10°)` via tf.data pipeline, training only |
| Epoch policy | 5-fold CV uses max 50 + early stopping; final retrain uses avg stopping epoch from CV |
| Feature cache | GLCM+HOG cached at `model_checkpoints/glcm_hog_features.npz` |
| Shared code | `dfu_common.py` — CONFIG, data loader, DFUModelTrainer, Optuna tuner |